# 05 — Greedy vs Beam Search: Flickr8k Caption Examples

Use your **best trained model from the Google Colab run**, **saved VGG16 features**, and the held-out Flickr8k **test images**. This notebook does not retrain the decoder.

**Algorithms**: Greedy = highest-probability next word; beam (k=3, k=5) = search over multiple hypotheses ranked by summed log probability, **without length normalization**.

Outputs include real generated captions, reference captions, comparison image grids and optional BLEU-1 through BLEU-4. Results do not exist until these cells are executed with your trained artifacts.

## 1. Configure Colab or local environment

On Colab, select a GPU runtime if available. The model is loaded from the **private Drive run folder** `MyDrive/NIT3251_captioning/runs/lstm_run1/`; the feature files live in `MyDrive/NIT3251_captioning/features/`.

**Important:** Do not commit `.keras` checkpoints or the Flickr8k dataset to GitHub. Local execution uses `models/lstm`, `data/features`, and `data/flickr8k`.

In [ ]:
from pathlib import Path
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/JustinDang69/Automatic-Image-Captioning-with-CNN-and-LSTM-.git"
REPO_BRANCH = "feature/greedy-beam-caption-examples"
RUN_NAME = "lstm_run1"
NUM_EXAMPLES = 6
BEAM_WIDTHS = (3, 5)
RANDOM_SEED = 42
RUN_FULL_BLEU = False  # True = evaluate all 810 held-out test images

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = Path("/content/nit3251-captioning")
    if not PROJECT_ROOT.exists():
        subprocess.run(["git", "clone", "--branch", REPO_BRANCH, REPO_URL,
                        str(PROJECT_ROOT)], check=True)
    else:
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "fetch", "origin"], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "checkout", REPO_BRANCH], check=True)
        subprocess.run(["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"], check=True)
    DRIVE_PROJECT = Path("/content/drive/MyDrive/NIT3251_captioning")
    MODEL_DIR = DRIVE_PROJECT / "runs" / RUN_NAME
    FEATURES_DIR = DRIVE_PROJECT / "features"
    OUTPUT_DIR = DRIVE_PROJECT / "runs" / RUN_NAME / "decoding_results"
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    MODEL_DIR = PROJECT_ROOT / "models" / "lstm"
    FEATURES_DIR = PROJECT_ROOT / "data" / "features"
    OUTPUT_DIR = MODEL_DIR / "decoding_results"

os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print("Project:", PROJECT_ROOT)
print("Trained model folder:", MODEL_DIR)
print("Feature folder:", FEATURES_DIR)

## 2. Import decoder helpers and locate Flickr8k captions

On Colab the dataset may already be present in Drive; if not, `kagglehub` downloads the Flickr8k image/captions archive. The trained model and VGG16 feature files must already be present.

In [ ]:
import json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from src.caption_decoding import (
    load_assets, greedy_decode, beam_search_decode, decode_image,
    example_captions, save_example_figure, evaluate_bleu
)

if IN_COLAB:
    drive_data = DRIVE_PROJECT / "flickr8k"
    if (drive_data / "Images").is_dir() and (drive_data / "captions.txt").exists():
        DATA_DIR = drive_data
    else:
        import kagglehub
        DATA_DIR = Path(kagglehub.dataset_download("adityajn105/flickr8k"))
else:
    DATA_DIR = PROJECT_ROOT / "data" / "flickr8k"

print("Flickr8k:", DATA_DIR)
print("Checkpoint:", (MODEL_DIR / "caption_model.keras").exists())
print("Saved VGG16 features:", (FEATURES_DIR / "features.npy").exists())

## 3. Load the trained model and exact tokenizer/splits

This step loads `caption_model.keras` (the best checkpoint copied at the end of training), `tokenizer.json`, `metadata.json`, `splits.json`, `features.npy` and `captions.txt`.

The vocabulary and test-image IDs **must match the training run**. The CNN weights do not need to be loaded for these already-extracted test images.

In [ ]:
assets = load_assets(
    model_dir=MODEL_DIR,
    features_dir=FEATURES_DIR,
    data_dir=DATA_DIR
)
print("Model inputs:", assets.model.input_shape)
print("Model output:", assets.model.output_shape)
print("Caption length:", assets.max_length)
print("Test images:", len(assets.splits["test"]))
print("Vocabulary size:", assets.tokenizer.vocab_size)

## 4. Generate actual greedy and beam-search captions

Each model call produces next-word probabilities. **Greedy** selects the highest-probability valid token. **Beam search** retains the top `k` hypotheses by cumulative log probability until `endseq` or the maximum length.

The same **trained weights** and **sampled test images** are used for both. Different results cannot be guaranteed: in some cases greedy and beam captions will be identical.

In [ ]:
examples = example_captions(
    assets, num_examples=NUM_EXAMPLES, seed=RANDOM_SEED,
    beam_widths=BEAM_WIDTHS, split="test"
)
for ex in examples:
    print("\\nIMAGE:", ex["image_id"])
    print("Reference:", ex["references"][0])
    print("Greedy   :", ex["greedy"])
    for k in BEAM_WIDTHS:
        print(f"Beam k={k}:", ex[f"beam_{k}"])

## 5. Display example images and human reference captions

This creates a figure with the image on the left and its **five human reference captions**, greedy caption, and beam captions on the right. Use these images in the **Results** and **Error Analysis** sections of the report.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
comparison_image = OUTPUT_DIR / "caption_examples.png"
save_example_figure(assets, examples, comparison_image)

from PIL import Image
with Image.open(comparison_image) as result_image:
    display(result_image.copy())

with (OUTPUT_DIR / "caption_examples.json").open("w", encoding="utf-8") as f:
    json.dump(examples, f, indent=2)
print("Saved:", comparison_image)
print("Saved:", OUTPUT_DIR / "caption_examples.json")

## 6. Compare caption examples in a table

Look for factual mistakes, hallucinated objects, missing actions, repetition, vague captions, and differences between greedy/beam outputs. The generated captions below are read from the trained model; no sample outputs are fabricated.

In [ ]:
rows = [{
    "Image": e["image_id"],
    "Reference (first of five)": e["references"][0],
    "Greedy": e["greedy"],
    "Beam (k=3)": e.get("beam_3", ""),
    "Beam (k=5)": e.get("beam_5", "")
} for e in examples]
display(pd.DataFrame(rows))

## 7. Optional: calculate BLEU-1 through BLEU-4 on the complete test set

Set `RUN_FULL_BLEU=True` in the first cell when you want a final evaluation. This can take some time because autoregressive caption generation makes many model calls.

The evaluation uses **all five reference captions** per image and reports **corpus-level BLEU** (NLTK). It also records average caption-generation time. Greedy and both beam settings use the *same* held-out test set. A smaller test subset should be labelled **exploratory**, not final.

In [ ]:
if RUN_FULL_BLEU:
    from nltk.translate.bleu_score import corpus_bleu
    scores = evaluate_bleu(assets, beam_widths=BEAM_WIDTHS)
    display(pd.DataFrame(scores["scores"]).T)
    (OUTPUT_DIR / "bleu_comparison.json").write_text(
        json.dumps(scores, indent=2), encoding="utf-8"
    )
    print("Full test set:", scores["full_test_set"])
else:
    print("Full BLEU evaluation skipped. Set RUN_FULL_BLEU=True and rerun.")

## 8. How to explain results in your report

- **Model**: frozen VGG16 fc2 encoder → Dense(256), plus Embedding(256) → LSTM(256), added and passed through Dense(256) and vocabulary softmax.
- **Decoding**: greedy searches one sequence; beam k=3 / k=5 retains multiple candidate sequences.
- **Main comparison**: BLEU-1 to BLEU-4, generation time, and human evaluation of image-grounding.
- **Possible limitation**: Standard beam search maximizes total log probability without a length penalty, so it often prefers shorter captions. If needed, length-normalized beam search is a separate experimental setting and should be labelled as such.
- **Valid results**: Use only captions generated from your trained model and evaluate using the saved Flickr8k *test split*. Do not use validation/test captions to fine-tune the decoder.

**Save for submission:** `caption_examples.png`, `caption_examples.json`, `bleu_comparison.json` (once evaluated), and screenshots showing concrete successes and failures.